In [60]:
import time
import os, random
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.stats import pearsonr
from sklearn.metrics import r2_score, mean_absolute_error

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [61]:
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [62]:
CHECKPOINT_PATH = "checkpoints_baseline/papagei_s.pt"
SIGNAL_PATH     = "itw_Left_data_stationary.npy"
LABEL_PATH      = "itw_HR_labels_data_stationary.npy"

In [63]:
ACTIVITIES  = ["stationary"]
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE  = 32
EPOCHS      = 50
LR          = 1e-3
CKPT_DIR    = "checkpoints/stationary"
FIG_DIR     = "figures"

In [64]:
# Subject splits 
TRAIN_SUBS = [f'S{i:02d}' for i in range(1,  22)]   # S01–S21
VAL_SUBS   = [f'S{i:02d}' for i in range(22, 26)]   # S22–S25
TEST_SUBS  = [f'S{i:02d}' for i in range(26, 31)]   # S26–S30

In [65]:
# PapaGei model config (must match checkpoint exactly) 
MODEL_CFG = dict(base_filters=32, kernel_size=3, stride=2,
                 groups=1, n_block=18, n_classes=512, n_experts=3)
EMBED_DIM  = 512 

In [66]:
ACT_COLOR = {
    "stationary":"#1f77b4",
}
ACT_LABEL = {
    "stationary":"Still"
}

for d in [CKPT_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Device : {DEVICE}")

Device : cpu


Papagei Foundational Model

In [67]:
# ResNet1DMoE Architecture .

class MyConv1dPadSame(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, groups=1):
        super().__init__()
        self.in_channels  = in_channels
        self.out_channels = out_channels
        self.kernel_size  = kernel_size
        self.stride       = stride
        self.groups       = groups
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size,
                              stride=stride, groups=groups)

    def forward(self, x):
        in_dim  = x.shape[-1]
        out_dim = (in_dim + self.stride - 1) // self.stride
        p       = max(0, (out_dim - 1) * self.stride + self.kernel_size - in_dim)
        x = F.pad(x, (p // 2, p - p // 2), "constant", 0)
        return self.conv(x)


class MyMaxPool1dPadSame(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.kernel_size = kernel_size
        self.stride      = 1
        self.max_pool    = nn.MaxPool1d(kernel_size=kernel_size)

    def forward(self, x):
        in_dim  = x.shape[-1]
        out_dim = (in_dim + self.stride - 1) // self.stride
        p       = max(0, (out_dim - 1) * self.stride + self.kernel_size - in_dim)
        x = F.pad(x, (p // 2, p - p // 2), "constant", 0)
        return self.max_pool(x)


class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride,
                 groups, downsample, use_bn, use_do, is_first_block=False):
        super().__init__()
        self.in_channels    = in_channels
        self.out_channels   = out_channels
        self.kernel_size    = kernel_size
        self.stride         = stride
        self.groups         = groups
        self.downsample     = downsample
        self.stride         = stride if downsample else 1
        self.is_first_block = is_first_block
        self.use_bn = use_bn
        self.use_do = use_do

        self.bn1   = nn.BatchNorm1d(in_channels)
        self.relu1 = nn.ReLU()
        self.do1   = nn.Dropout(p=0.5)
        self.conv1 = MyConv1dPadSame(in_channels, out_channels,
                                     kernel_size, self.stride, groups)
        self.bn2   = nn.BatchNorm1d(out_channels)
        self.relu2 = nn.ReLU()
        self.do2   = nn.Dropout(p=0.5)
        self.conv2 = MyConv1dPadSame(out_channels, out_channels,
                                     kernel_size, 1, groups)
        self.max_pool = MyMaxPool1dPadSame(kernel_size=self.stride)

    def forward(self, x):
        identity = x
        out = x
        if not self.is_first_block:
            if self.use_bn: out = self.bn1(out)
            out = self.relu1(out)
            if self.use_do: out = self.do1(out)
        out = self.conv1(out)
        if self.use_bn: out = self.bn2(out)
        out = self.relu2(out)
        if self.use_do: out = self.do2(out)
        out = self.conv2(out)
        if self.downsample:
            identity = self.max_pool(identity)
        if self.out_channels != self.in_channels:
            identity = identity.transpose(-1, -2)
            ch1 = (self.out_channels - self.in_channels) // 2
            ch2 = self.out_channels - self.in_channels - ch1
            identity = F.pad(identity, (ch1, ch2), "constant", 0)
            identity = identity.transpose(-1, -2)
        out += identity
        return out


class ResNet1DMoE(nn.Module):
   
    def __init__(self, in_channels, base_filters, kernel_size, stride,
                 groups, n_block, n_classes, n_experts=3,
                 downsample_gap=2, increasefilter_gap=4,
                 use_bn=True, use_do=True, verbose=False, use_projection=False):
        super().__init__()
        self.verbose            = verbose
        self.n_block            = n_block
        self.kernel_size        = kernel_size
        self.stride             = stride
        self.groups             = groups
        self.use_bn             = use_bn
        self.use_do             = use_do
        self.use_projection     = use_projection
        self.downsample_gap     = downsample_gap
        self.increasefilter_gap = increasefilter_gap
        self.n_experts          = n_experts

        self.first_block_conv = MyConv1dPadSame(in_channels, base_filters,
                                                kernel_size, stride=1)
        self.first_block_bn   = nn.BatchNorm1d(base_filters)
        self.first_block_relu = nn.ReLU()
        out_channels = base_filters

        self.basicblock_list = nn.ModuleList()
        for i_block in range(n_block):
            is_first   = (i_block == 0)
            downsample = (i_block % downsample_gap == 1)
            in_ch  = base_filters if is_first else int(
                base_filters * 2 ** ((i_block - 1) // increasefilter_gap))
            out_ch = in_ch * 2 if (
                i_block % increasefilter_gap == 0 and i_block != 0) else in_ch
            self.basicblock_list.append(
                BasicBlock(in_ch, out_ch, kernel_size, stride,
                           groups, downsample, use_bn, use_do,
                           is_first_block=is_first))
            out_channels = out_ch

        if use_projection:
            self.projector = nn.Sequential(
                nn.Linear(out_channels, 256), nn.BatchNorm1d(256),
                nn.ReLU(), nn.Linear(256, 128))

        self.final_bn   = nn.BatchNorm1d(out_channels)
        self.final_relu = nn.ReLU(inplace=True)
        self.dense      = nn.Linear(out_channels, n_classes)

        self.expert_layers_1 = nn.ModuleList([
            nn.Sequential(nn.Linear(out_channels, out_channels // 2),
                          nn.ReLU(),
                          nn.Linear(out_channels // 2, 1))
            for _ in range(n_experts)])
        self.gating_network_1 = nn.Sequential(
            nn.Linear(out_channels, n_experts), nn.Softmax(dim=1))

        self.expert_layers_2 = nn.ModuleList([
            nn.Sequential(nn.Linear(out_channels, out_channels // 2),
                          nn.ReLU(), nn.Dropout(0.3),
                          nn.Linear(out_channels // 2, 1))
            for _ in range(n_experts)])
        self.gating_network_2 = nn.Sequential(
            nn.Linear(out_channels, n_experts), nn.Softmax(dim=1))

    def forward(self, x):
        out = x
        out = self.first_block_conv(out)
        if self.use_bn: out = self.first_block_bn(out)
        out = self.first_block_relu(out)
        for blk in self.basicblock_list:
            out = blk(out)
        if self.use_bn: out = self.final_bn(out)
        out = self.final_relu(out)
        out = out.mean(-1)                      # (B, 512) - embedding

        out_class = self.dense(out)

        expert_out1   = torch.stack([e(out) for e in self.expert_layers_1], dim=1)
        gate1         = self.gating_network_1(out)
        out_moe1      = torch.sum(gate1.unsqueeze(2) * expert_out1, dim=1)

        expert_out2   = torch.stack([e(out) for e in self.expert_layers_2], dim=1)
        gate2         = self.gating_network_2(out)
        out_moe2      = torch.sum(gate2.unsqueeze(2) * expert_out2, dim=1)

        return out_class, out_moe1, out_moe2, out   # out = 512-dim embedding

In [68]:
def load_papagei_backbone(ckpt_path: str) -> ResNet1DMoE:
    
    model = ResNet1DMoE(
        in_channels  = 1,
        base_filters = MODEL_CFG["base_filters"],
        kernel_size  = MODEL_CFG["kernel_size"],
        stride       = MODEL_CFG["stride"],
        groups       = MODEL_CFG["groups"],
        n_block      = MODEL_CFG["n_block"],
        n_classes    = MODEL_CFG["n_classes"],
        n_experts    = MODEL_CFG["n_experts"],
    )

    raw_state = torch.load(ckpt_path, map_location="cpu")

    # Strip 'module.' prefix from DataParallel checkpoint
    state = {}
    for k, v in raw_state.items():
        new_k = k[len("module."):] if k.startswith("module.") else k
        state[new_k] = v

    missing, unexpected = model.load_state_dict(state, strict=True)
    if missing:
        print(f"  Missing keys   : {missing[:4]}")
    if unexpected:
        print(f"  Unexpected keys: {unexpected[:4]}")


    n_total = sum(p.numel() for p in model.parameters())
    print(f"  Backbone loaded : {ckpt_path}")
    print(f"  Total params : {n_total:,}", flush=True)
    return model


BACKBONE = load_papagei_backbone(CHECKPOINT_PATH)
BACKBONE.to(DEVICE)


  Backbone loaded : checkpoints_baseline/papagei_s.pt
  Total params : 5,785,612


ResNet1DMoE(
  (first_block_conv): MyConv1dPadSame(
    (conv): Conv1d(1, 32, kernel_size=(3,), stride=(1,))
  )
  (first_block_bn): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (first_block_relu): ReLU()
  (basicblock_list): ModuleList(
    (0): BasicBlock(
      (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu1): ReLU()
      (do1): Dropout(p=0.5, inplace=False)
      (conv1): MyConv1dPadSame(
        (conv): Conv1d(32, 32, kernel_size=(3,), stride=(1,))
      )
      (bn2): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu2): ReLU()
      (do2): Dropout(p=0.5, inplace=False)
      (conv2): MyConv1dPadSame(
        (conv): Conv1d(32, 32, kernel_size=(3,), stride=(1,))
      )
      (max_pool): MyMaxPool1dPadSame(
        (max_pool): MaxPool1d(kernel_size=1, stride=1, padding=0, dilation=1, ceil_mode=False)
      )
    )
    (1): BasicBlock(
      (bn1): Bat

In [ ]:
class PapaGeiLinearHead(nn.Module):

    def __init__(self, backbone: ResNet1DMoE, embed_dim: int = EMBED_DIM):
        super().__init__()
        self.backbone = backbone
        self.head     = nn.Linear(embed_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
      
        if x.dim() == 2:
            x = x.unsqueeze(1)          # (B, T) - (B, 1, T)
            
        _, _, _, emb = self.backbone(x)   # emb: (B, 512)

        return self.head(emb).squeeze(-1)     # (B,)


In [70]:
class PPGHRDataset(Dataset):
    def __init__(self, signals: np.ndarray, labels: np.ndarray):
        self.signals = torch.tensor(signals, dtype=torch.float32)  # (N, 1250)
        self.labels  = torch.tensor(labels,  dtype=torch.float32)  # (N,)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.signals[idx], self.labels[idx]

In [71]:
_CACHE = None

def load_all_data(force: bool = False) -> dict:
    """Load and align signal + label files. Module-level cache."""
    global _CACHE
    if _CACHE is not None and not force:
        return _CACHE

    print("  Loading signal file …", flush=True)
    sig   = np.load(SIGNAL_PATH,  allow_pickle=True).item()
    print("  Loading label  file …", flush=True)
    label = np.load(LABEL_PATH,   allow_pickle=True).item()

    s_ids = sig["trial_ids"]
    l_ids = label["trial_ids"]

    if np.array_equal(s_ids, l_ids):
        print("  Alignment: exact match ", flush=True)
        merged = {
            "signal"      : sig["signal"],          # (N, 1250) float32
            "HR"          : label["HR"],             # (N,)
            "subject_ids" : sig["subject_ids"],
            "activities"  : sig["activities"],
            "trial_ids"   : s_ids,
        }
    else:
        print("  Alignment: remapping by trial_id …", flush=True)
        hr_map = dict(zip(l_ids, label["HR"]))
        hr_arr = np.array([hr_map.get(t, np.nan) for t in s_ids], dtype=np.float32)
        valid  = ~np.isnan(hr_arr)
        merged = {
            "signal"      : sig["signal"][valid],
            "HR"          : hr_arr[valid],
            "subject_ids" : sig["subject_ids"][valid],
            "activities"  : sig["activities"][valid],
            "trial_ids"   : s_ids[valid],
        }
    print(f"  Total: {len(merged['HR']):,} segs  "
          f"HR {merged['HR'].min():.1f}–{merged['HR'].max():.1f} bpm", flush=True)
    _CACHE = merged
    return merged


def get_splits(all_data: dict, activity: str):
    mask  = all_data["activities"] == activity
    sigs  = all_data["signal"][mask]
    hr    = all_data["HR"][mask]
    subjs = all_data["subject_ids"][mask]
    if len(sigs) == 0:
        raise ValueError(f"No data for '{activity}'")

    def pick(sub_list):
        m = np.isin(subjs, sub_list)
        return sigs[m], hr[m], subjs[m]

    X_tr,  y_tr,  _        = pick(TRAIN_SUBS)
    X_val, y_val, _        = pick(VAL_SUBS)
    X_te,  y_te,  subj_te  = pick(TEST_SUBS)
    print(f"  [{activity:>12}]  "
          f"train={len(X_tr):4d}  val={len(X_val):4d}  test={len(X_te):4d}",
          flush=True)
    return (X_tr, y_tr), (X_val, y_val), (X_te, y_te, subj_te)


def compute_metrics(truths: np.ndarray, preds: np.ndarray) -> dict:
    sq  = (truths - preds) ** 2
    r,_ = pearsonr(truths, preds)
    return {
        "mae"    : mean_absolute_error(truths, preds),
        "mae_std": np.abs(truths - preds).std(),
        "mse"    : sq.mean(),
        "mse_std": sq.std(),
        "r"      : r,
        "r2"     : r2_score(truths, preds),
    }


In [72]:

def train_activity(activity: str, all_data: dict) -> dict:
  
    head_ckpt = os.path.join(CKPT_DIR, f"{activity}.pt")
    print(f"\n{'═'*60}", flush=True)
    print(f"  Activity : {activity.upper()}", flush=True)
    print(f"  Loss     : Huber  |  Backbone: Papagei", flush=True)
    print(f"  Epochs   : {EPOCHS}  LR={LR}  Batch={BATCH_SIZE}", flush=True)
    print(f"{'═'*60}", flush=True)

    (X_tr, y_tr), (X_val, y_val), (X_te, y_te, subj_te) = \
        get_splits(all_data, activity)

    tr_loader  = DataLoader(PPGHRDataset(X_tr,  y_tr),
                            BATCH_SIZE, shuffle=True,  num_workers=0)
    val_loader = DataLoader(PPGHRDataset(X_val, y_val),
                            BATCH_SIZE, shuffle=False, num_workers=0)
    te_loader  = DataLoader(PPGHRDataset(X_te,  y_te),
                            BATCH_SIZE, shuffle=False, num_workers=0)

    model     = PapaGeiLinearHead(BACKBONE, EMBED_DIM).to(DEVICE)
    criterion = nn.SmoothL1Loss()

    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    best_val  = float("inf")
    history   = {"train": [], "val": []}

    for epoch in range(1, EPOCHS + 1):

        # Train 
        model.train()          
        tr_loss = 0.0
        for X_b, y_b in tr_loader:
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
            loss = criterion(model(X_b), y_b)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            tr_loss += loss.item() * len(y_b)
        tr_loss /= len(tr_loader.dataset)

        # Validate 
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_b, y_b in val_loader:
                X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
                val_loss += criterion(model(X_b), y_b).item() * len(y_b)
        val_loss /= len(val_loader.dataset)

        scheduler.step()
        history["train"].append(tr_loss)
        history["val"].append(val_loss)

        tag = ""
        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), head_ckpt)
            tag = "  - best saved"

        # Real-time print 
        print(
            f"  [{activity}] ep {epoch:02d}/{EPOCHS}"
            f"  train_MAE={tr_loss:.3f}"
            f"  val_MAE={val_loss:.3f}"
            f"  best={best_val:.3f}{tag}",
            flush=True
        )

    # Test evaluation 
    model.load_state_dict(torch.load(head_ckpt, map_location=DEVICE))
    model.eval()
    all_p, all_t = [], []
    with torch.no_grad():
        for X_b, y_b in te_loader:
            all_p.append(model(X_b.to(DEVICE)).cpu().numpy())
            all_t.append(y_b.numpy())
    preds  = np.concatenate(all_p)
    truths = np.concatenate(all_t)
    print("\n  ── Truth/Pred Statistics ──")
    print("  Truth mean:", truths.mean(), "std:", truths.std())
    print("  Pred  mean:", preds.mean(), "std:", preds.std())
    m      = compute_metrics(truths, preds)

    print(f"\n  ── Test [{activity}] ──────────────────────────────────", flush=True)
    print(f"     MAE       : {m['mae']:.3f} ± {m['mae_std']:.3f} bpm",  flush=True)
    print(f"     Pearson r : {m['r']:.4f}",                              flush=True)
    print(f"     R²        : {m['r2']:.4f}",                             flush=True)

    return {
        "activity"     : activity,
        "truths"       : truths,
        "preds"        : preds,
        "subjects"     : subj_te,
        "metrics"      : m,
        "history"      : history,
        "window_errors": np.abs(truths - preds),
    }


In [73]:
def plot_per_activity_scatter(results: list, save_dir: str = FIG_DIR):
    n = len(results)
    fig, axes = plt.subplots(1, n, figsize=(15, 10))
    axes = np.array(axes).flatten()
    for i, r in enumerate(results):
        ax = axes[i]; m = r["metrics"]
        c  = ACT_COLOR.get(r["activity"], "steelblue")
        t, p = r["truths"], r["preds"]
        lo = min(t.min(), p.min()) - 5
        hi = max(t.max(), p.max()) + 5
        ax.scatter(t, p, alpha=0.45, s=14, color=c, edgecolors="none")
        ax.plot([lo, hi], [lo, hi], "k-", lw=1.5, zorder=5)
        ax.text(0.05, 0.95,
                f"r = {m['r']:.3f}\n"
                f"R² = {m['r2']:.3f}\n"
                f"MAE = {m['mae']:.2f} ± {m['mae_std']:.2f} bpm\n"
                f"MSE = {m['mse']:.2f} ± {m['mse_std']:.2f}",
                transform=ax.transAxes, fontsize=8.5, va="top",
                bbox=dict(boxstyle="round,pad=0.3",
                          facecolor="white", alpha=0.85, edgecolor="gray"))
        ax.set(title=ACT_LABEL.get(r["activity"], r["activity"]),
               xlabel="Ground Truth HR (bpm)", ylabel="Predicted HR (bpm)",
               xlim=(lo, hi), ylim=(lo, hi))
        ax.set_aspect("equal", adjustable="box")
        ax.grid(True, linestyle="--", alpha=0.4)
    plt.suptitle("Per-Activity HR Prediction — Pearson Correlation",
                 fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()
    p = os.path.join(save_dir, "scatter_per_activity.png")
    plt.savefig(p, dpi=300, bbox_inches="tight"); plt.close()
    print(f"  Saved: {p}", flush=True)


def plot_continuous_tracking(results: list, target: str = "S26",
                             save_dir: str = FIG_DIR):
    fig, ax = plt.subplots(figsize=(15, 4))
    t_all, p_all, bounds = [], [], []
    idx = 0
    for r in results:
        mask = r["subjects"] == target
        t_, p_ = r["truths"][mask], r["preds"][mask]
        if len(t_) == 0:
            print(f"  [tracking] no data for {target} in {r['activity']}", flush=True)
            continue
        t_all.extend(t_); p_all.extend(p_)
        idx += len(t_)
        bounds.append((ACT_LABEL.get(r["activity"], r["activity"]), idx))
    if not t_all:
        print(f"  No data for {target}", flush=True); return
    t_all = np.array(t_all); p_all = np.array(p_all)
    y_top = t_all.max() + 10
    ax.plot(t_all, label="Ground truth", color="#1f77b4", lw=1.5)
    ax.plot(p_all, label="Predicted",    color="#ff7f0e", lw=1.5, alpha=0.85)
    prev = 0
    for name, end in bounds:
        ax.axvline(x=end, color="lightgray", lw=1.0)
        ax.text(prev + (end - prev) / 2, y_top, name,
                ha="center", va="bottom", fontsize=11)
        prev = end
    ax.set(xlim=(0, len(t_all)), xlabel="Window", ylabel="BPM")
    ax.legend(loc="upper left", fontsize=10)
    ax.grid(True, linestyle="--", alpha=0.3)
    plt.tight_layout()
    p = os.path.join(save_dir, f"tracking_{target}.png")
    plt.savefig(p, dpi=300, bbox_inches="tight"); plt.close()
    print(f"  Saved: {p}", flush=True)


def plot_mae_table(results: list, save_dir: str = FIG_DIR):

    rows = []
    for r in results:
        m = r["metrics"]
        rows.append([ACT_LABEL.get(r["activity"], r["activity"]),
                     f"{m['mae']:.2f}", f"{m['mae_std']:.2f}"])

    fig, ax = plt.subplots(figsize=(5, 3))
    ax.axis("off")
    col_labels = ["Activity", "mean", "std"]
 
    tbl = ax.table(
        cellText=rows,
        colLabels=col_labels,
        cellLoc="center",
        loc="center",
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1.4, 1.6)

    # Style header
    for j in range(3):
        tbl[(0, j)].set_facecolor("#e8e8e8")
        tbl[(0, j)].set_text_props(fontweight="bold")

    # Bold activity column
    for i in range(1, len(rows) + 1):
        tbl[(i, 0)].set_text_props(fontstyle="italic")

    ax.set_title("MAE (bpm)", fontsize=11, fontweight="bold", pad=4)
    plt.tight_layout()
    p = os.path.join(save_dir, "metrics_table.png")
    plt.savefig(p, dpi=200, bbox_inches="tight"); plt.close()
    print(f"  Saved: {p}", flush=True)


def plot_training_curves(results: list, save_dir: str = FIG_DIR):
    n = len(results)
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()
    for i, r in enumerate(results):
        ax   = axes[i]
        hist = r["history"]
        ax.plot(hist["train"], label="Train MAE", color="steelblue")
        ax.plot(hist["val"],   label="Val MAE",   color="tomato")
        ax.set(title=ACT_LABEL.get(r["activity"], r["activity"]),
               xlabel="Epoch", ylabel="MAE (bpm)")
        ax.legend(fontsize=8)
        ax.grid(True, linestyle="--", alpha=0.4)
    plt.suptitle("Training Curves — Linear Head (PapaGei Backbone)",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    p = os.path.join(save_dir, "training_curves.png")
    plt.savefig(p, dpi=300, bbox_inches="tight"); plt.close()
    print(f"  Saved: {p}", flush=True)




In [74]:
if __name__ == "__main__":

    all_data    = load_all_data()
    all_results = []

    for activity in ACTIVITIES:
        res = train_activity(activity, all_data)
        all_results.append(res)

    # Figures 
    print(f"\n  Generating figures - {FIG_DIR}/", flush=True)
    plot_per_activity_scatter(all_results, save_dir=FIG_DIR)
    plot_continuous_tracking(all_results, target="S26")
    plot_continuous_tracking(all_results, target="S27")
    plot_mae_table(all_results)
    plot_training_curves(all_results)
    print("\n  All done", flush=True)


  Loading signal file …
  Loading label  file …
  Alignment: exact match 
  Total: 1,770 segs  HR 53.5–113.3 bpm

════════════════════════════════════════════════════════════
  Activity : STATIONARY
  Loss     : Huber  |  Backbone: Papagei
  Epochs   : 50  LR=0.001  Batch=32
════════════════════════════════════════════════════════════
  [  stationary]  train=1239  val= 236  test= 295
  [stationary] ep 01/50  train_MAE=76.833  val_MAE=59.085  best=59.085  - best saved
  [stationary] ep 02/50  train_MAE=66.245  val_MAE=48.333  best=48.333  - best saved
  [stationary] ep 03/50  train_MAE=51.788  val_MAE=38.162  best=38.162  - best saved
  [stationary] ep 04/50  train_MAE=33.920  val_MAE=14.614  best=14.614  - best saved
  [stationary] ep 05/50  train_MAE=15.230  val_MAE=9.863  best=9.863  - best saved
  [stationary] ep 06/50  train_MAE=7.136  val_MAE=5.974  best=5.974  - best saved
  [stationary] ep 07/50  train_MAE=5.295  val_MAE=3.215  best=3.215  - best saved
  [stationary] ep 08/50  t